# Mind Quest: from one player to a group

Mind Quest is an educational jump-and-run game about attention, response inhibition and executive control. While playing, a player completes three classic experiments: a **flanker task** (Eriksen & Eriksen, 1974; Rueda et al., 2004), a **stop-signal task** (Logan & Cowan, 1984; Verbruggen et al., 2019) and a **2-back task** (Kirchner, 1958), plus a short questionnaire about screen habits, gaming and alertness.

One playthrough tells us about one person. This notebook turns it into data a statistical analysis can work with:

1. **Play** the game: the experiments save the player's data in `data/`.
2. **Bootstrap** the player's trials: how precise are their scores?
3. **Simulate** a group of players whose typical scores come from the bootstrap and whose differences come from the literature.
4. **Analyse** the simulated group.

All code lives in `src/`; this notebook only calls its high-level functions. Run it from the project folder, from top to bottom.

## Setup

Install the requirements once with `pip install -r requirements.txt` (see `README.md`).

In [ ]:
from src.bootstrap import bootstrap_session, save_bootstrap
from src.data import clear_game_progress, load_session
from src.game import run_game
from src.simulation import SIMULATED_DIR, Effects, simulate_participants

## 1. Play the game

Running the cell below opens the game window. It starts with the questionnaire; after that the levels open one after the other. The experiments are in Level 1 (flanker), Level 2 (stop-signal) and Level 3 (2-back and Tower of London). Close the window to come back to the notebook; every finished experiment has been saved in `data/`.

* `PARTICIPANT_ID`: an anonymous label for the data files. `None` creates a random label such as `p-3fa2c1`.
* `NEW_PARTICIPANT = True` first clears the unlocked levels and ghost races of the previous player (the data in `data/` are kept).
* `PLAY_GAME = False` skips the game and uses the data of whoever played an experiment last. If nobody has (for example in a fresh clone), the example playthrough in `data/example/` is used.

Admin mode is switched off (`allow_admin=False`), so no experiment can be skipped.

In [ ]:
PARTICIPANT_ID = None
NEW_PARTICIPANT = True
PLAY_GAME = True

if PLAY_GAME:
    if NEW_PARTICIPANT:
        clear_game_progress()
    run_game(participant_id=PARTICIPANT_ID, allow_admin=False)

## 2. Bootstrap the player's data

Each score rests on a few dozen trials, so it would come out somewhat differently if the player played again. The bootstrap (Efron, 1979) estimates how much: the player's trials are drawn again **with replacement** (within each condition, so the design of the task stays intact), the score is recomputed, and this is repeated 5,000 times. The spread of these values is the standard error of the score, and their middle 95% is a confidence interval.

We use the most recent player who completed at least one of the flanker, stop-signal and 2-back tasks, their most recent session and their **first** attempt at every experiment (later attempts are affected by practice); practice trials are left out. The bootstrapped measures are:

| Task | Measures |
| --- | --- |
| Flanker | baseline RT (congruent trials), flanker effect (incongruent minus congruent RT; Eriksen & Eriksen, 1974), trial-to-trial noise, error rates |
| Stop-signal | mean go RT, its noise, choice errors, and the stop-signal reaction time (SSRT, integration method; Verbruggen et al., 2019) |
| 2-back | sensitivity d′ (log-linear correction; Hautus, 1995), response criterion *c* (Stanislaw & Todorov, 1999), hit RTs |

The SSRT is the least precise measure: the game has 16 stop trials, while Verbruggen et al. (2019) recommend at least 50. Tasks the player did not complete are skipped, and the simulation then uses published values instead. The standard errors are used in the next step: they decide how much the player's scores shape the simulated group.

The results are saved in `data/simulated/`: `bootstrap_summary.csv` (estimate, standard error and 95% CI per measure) and `bootstrap_samples.csv` (all 5,000 bootstrap values per measure).

In [ ]:
session = load_session(PARTICIPANT_ID)
participant = session["participant_id"]
completed = [task for task in session if task not in ("participant_id", "source")]
print(f"Player: {participant} (from {session['source']}); completed: {', '.join(completed) or 'nothing'}")

boot = bootstrap_session(session, n_resamples=5000, seed=2026)
for path in save_bootstrap(boot, participant) or ["Nothing to bootstrap: the simulation will use published values only."]:
    print(path)

## 3. Simulate a group of players

Every simulated player answers the questionnaire and completes every trial of the flanker, stop-signal and 2-back tasks, in exactly the format the game writes, so the same analysis code works on real and simulated data. Each player first gets their own true scores, drawn from normal distributions:

| Score | Mean | How much people differ (SD) |
| --- | --- | --- |
| Flanker baseline RT and flanker effect | player and Kaufman et al. (2016), weighted by precision | Kaufman et al. (2016), Table 2, young adults |
| Go RT and SSRT | player and Congdon et al. (2014), weighted by precision | Congdon et al. (2014), Table 1, healthy controls |
| 2-back d′ | player and Forsberg et al. (2020), weighted by precision | Forsberg et al. (2020), Table 3, younger adults |
| Daily screen time | Sedaghat et al. (2025) | Sedaghat et al. (2025) |

**How the player and the literature are combined.** The player's score is one noisy measurement; the published mean describes the population. Following the normal model with known variances (Gelman et al., 2013, Section 2.5), both are averaged with weights proportional to their precision (1 / variance). The player's weight is SD² / (SD² + SE²), with SD the between-person SD from the literature and SE the bootstrap standard error. A precisely measured player (small SE) moves the group towards their own score; a noisy one (large SE) leaves it close to the published mean. The weights are saved with the other values in `parameters_simulated.csv` (`player_weight`). For tasks the player did not complete, the published means are used. Trial-to-trial noise and error rates have no published between-person SD and take the player's own estimate.

**The group is based on one player's session.** The simulated players are centred on one person's scores (as far as their precision allows), not on a sample of real people. They show what a group *could* look like, which is useful for planning and checking an analysis, but they are no substitute for collecting data from many players.

Values without a published source (error rates, the 2-back criterion and RTs, and the answer distributions of the 1–5 questions) are marked as assumptions in `src/simulation.py`. Screen time comes from a study that asked about all screen use, including study and work, while the game asks about free time only, so the simulated screen times are probably too high.

The trials then follow a model of each task: RT = baseline + flanker effect + noise in the flanker task, a race between going and stopping in the stop-signal task (Logan & Cowan, 1984), and signal detection theory in the 2-back task (Stanislaw & Todorov, 1999).

**The effects below are assumptions, not findings.** By default every hour of screen time above the mean adds 5 ms to the flanker effect and to the SSRT, and higher alertness speeds up responses. A simulation only contains the effects that are put into it: it is a tool to plan and check an analysis, not evidence that the effects exist. Set an effect to 0 to simulate a world without it.

Everything is saved in `data/simulated/`: one file per task (`questionnaire_simulated.csv`, `flanker_simulated.csv`, `stopsignal_simulated.csv`, `nback_simulated.csv`), the true scores of every simulated player (`true_values_simulated.csv`) and the population values used, with their sources (`parameters_simulated.csv`).

In [ ]:
effects = Effects(
    screen_time_on_flanker_ms=5.0,
    screen_time_on_ssrt_ms=5.0,
    alertness_on_rt_ms=-15.0,
)
sim = simulate_participants(n=50, bootstrap=boot, participant_id=participant, seed=2026, effects=effects)
for path in sim.save(SIMULATED_DIR):
    print(path)

## 4. Statistical analysis

*To be added.* The simulated players are in `data/simulated/` and also available here as `sim` (`sim.questionnaire`, `sim.flanker`, `sim.stopsignal`, `sim.nback`, `sim.true_values`). Real data can be loaded with `src.data.load_task`, for example `load_task("flanker")`.

## References

Congdon, E., Altshuler, L. L., Mumford, J. A., Karlsgodt, K. H., Sabb, F. W., Ventura, J., McGough, J. J., London, E. D., Cannon, T. D., Bilder, R. M., & Poldrack, R. A. (2014). Neural activation during response inhibition in adult attention-deficit/hyperactivity disorder: Preliminary findings on the effects of medication and symptom severity. *Psychiatry Research: Neuroimaging, 222*, 17–28. https://doi.org/10.1016/j.pscychresns.2014.02.002

Efron, B. (1979). Bootstrap methods: Another look at the jackknife. *The Annals of Statistics, 7*(1), 1–26. https://doi.org/10.1214/aos/1176344552

Eriksen, B. A., & Eriksen, C. W. (1974). Effects of noise letters upon the identification of a target letter in a nonsearch task. *Perception & Psychophysics, 16*(1), 143–149.

Forsberg, A., Fellman, D., Laine, M., Johnson, W., & Logie, R. H. (2020). Strategy mediation in working memory training in younger and older adults. *Quarterly Journal of Experimental Psychology, 73*(8), 1206–1226. https://doi.org/10.1177/1747021820915107

Gelman, A., Carlin, J. B., Stern, H. S., Dunson, D. B., Vehtari, A., & Rubin, D. B. (2013). *Bayesian data analysis* (3rd ed.). CRC Press.

Hautus, M. J. (1995). Corrections for extreme proportions and their biasing effects on estimated values of d'. *Behavior Research Methods, Instruments, & Computers, 27*(1), 46–51.

Kaufman, D. A. S., Sozda, C. N., Dotson, V. M., & Perlstein, W. M. (2016). An event-related potential investigation of the effects of age on alerting, orienting, and executive function. *Frontiers in Aging Neuroscience, 8*, 99. https://doi.org/10.3389/fnagi.2016.00099

Kirchner, W. K. (1958). Age differences in short-term retention of rapidly changing information. *Journal of Experimental Psychology, 55*(4), 352–358.

Logan, G. D., & Cowan, W. B. (1984). On the ability to inhibit thought and action: A theory of an act of control. *Psychological Review, 91*(3), 295–327.

Rueda, M. R., Fan, J., McCandliss, B. D., Halparin, J. D., Gruber, D. B., Lercari, L. P., & Posner, M. I. (2004). Development of attentional networks in childhood. *Neuropsychologia, 42*(8), 1029–1040.

Sedaghat, D., Lee, E., Garcia, J., & Brazendale, K. (2025). Screen time and objectively measured sleep of U.S. college students: A brief report. *International Journal of Exercise Science, 18*(4), 747–756. https://doi.org/10.70252/XDUT8341

Stanislaw, H., & Todorov, N. (1999). Calculation of signal detection theory measures. *Behavior Research Methods, Instruments, & Computers, 31*(1), 137–149. https://doi.org/10.3758/BF03207704

Verbruggen, F., Aron, A. R., Band, G. P. H., Beste, C., Bissett, P. G., Brockett, A. T., et al. (2019). A consensus guide to capturing the ability to inhibit actions and impulsive behaviors in the stop-signal task. *eLife, 8*, e46323.

The game's own content (the insights shown while playing) has further sources; they are listed in `README.md` and in the game under "References".